# SP02 — Spatial Domain Detection

**Tools:** `banksy-py`, `GraphST`  
**Dataset:** Mouse brain Visium H&E (`../data/visium_processed.h5ad` from SP01)  
**Papers:** [Singhal et al. 2024, Nature Genetics (BANKSY)](https://doi.org/10.1038/s41588-024-01664-3) · [Long et al. 2023, Nature Methods (GraphST)](https://doi.org/10.1038/s41592-023-01671-z)

**Prerequisites:** SP01 (Visium preprocessing). `../data/visium_processed.h5ad` required.

---

## Why expression-only clustering fails for spatial domains

Standard Leiden clustering on gene expression captures **cell-type identity** — CD4+ T, B cell, etc. But in a tissue section, you often want **tissue regions** (cortical layers, white matter, hippocampal CA1/CA3) that are defined by *spatial context*, not just intrinsic identity.

The problem: two oligodendrocytes in white matter and one oligodendrocyte that has migrated to cortex have identical transcriptomes but belong to different tissue contexts.

## The spatial domain detection idea

All methods solve the same problem: augment each spot/cell's feature vector with information from its **spatial neighbors** before clustering.

```
Expression-only:   z_i = x_i
Spatially-aware:   z_i = (1-λ)·x_i  +  λ·mean(x_neighbors(i))
                            ↑ own expression  ↑ neighborhood expression avg
```

λ (lambda) controls the balance. λ=0 is pure expression-only Leiden; λ=1 is pure spatial smoothing.

**BANKSY** implements this idea with learned neighborhood aggregation weights (similar to graph convolution). **GraphST** uses a graph attention network to learn the spatial encoding.

| Method | Architecture | Speed | Strengths |
|--------|-------------|-------|----------|
| **BANKSY** | Weighted spatial avg + NMF/PCA | Fast | Interpretable λ; multi-scale neighborhoods |
| **GraphST** | Graph attention encoder (GNN) | Moderate | Handles non-uniform spot densities; multi-sample |
| **STAGATE** | Graph attention autoencoder | Moderate | Denoising + domain detection jointly |
| **BayesSpace** | Markov Random Field | Slow | Bayesian; requires R |

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt

sc.settings.set_figure_params(dpi=80, facecolor='white')

## 1. Load Data

In [ ]:
import os
if os.path.exists('../data/visium_processed.h5ad'):
    adata = sc.read_h5ad('../data/visium_processed.h5ad')
else:
    # Rebuild from SP01 if needed
    adata = sq.datasets.visium_hne_adata()
    sc.pp.filter_genes(adata, min_cells=10)
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=3000)
    sc.pp.pca(adata)
    sc.pp.neighbors(adata)
    sc.tl.umap(adata)
    sc.tl.leiden(adata, resolution=0.4)
    sq.gr.spatial_neighbors(adata, coord_type='visium', n_neighs=6)

print(adata)

In [ ]:
# Baseline: expression-only Leiden
if 'leiden' not in adata.obs:
    sc.tl.leiden(adata, resolution=0.4)
    
sq.pl.spatial_scatter(adata, color='leiden', size=1.5, figsize=(5, 5),
                       title='Baseline: expression-only Leiden')

## 2. Manual BANKSY-Style Spatial Augmentation

Before using the full BANKSY package, let's implement the core idea manually using squidpy's spatial graph — this makes the mechanism transparent.

In [ ]:
# The spatial connectivity matrix is already in adata.obsp['spatial_connectivities']
# (built by sq.gr.spatial_neighbors)
if 'spatial_connectivities' not in adata.obsp:
    sq.gr.spatial_neighbors(adata, coord_type='visium', n_neighs=6)

W = adata.obsp['spatial_connectivities'].toarray()
# Row-normalize: each row sums to 1 → weighted average of neighbors
row_sums = W.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
W_norm = W / row_sums

# Get expression matrix (HVG PCA scores)
X = adata.obsm['X_pca'][:, :30]  # first 30 PCs

# Augment: z_i = (1-lambda)*x_i + lambda * mean(x_neighbors)
for lam in [0.0, 0.2, 0.5, 0.8]:
    X_aug = (1 - lam) * X + lam * (W_norm @ X)
    adata.obsm[f'X_banksy_lam{int(lam*10)}'] = X_aug
    print(f'Lambda={lam}: augmented shape {X_aug.shape}')

In [ ]:
# Cluster at each lambda and compare spatial coherence
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
lambdas = [0.0, 0.2, 0.5, 0.8]
for ax, lam in zip(axes, lambdas):
    rep_key = f'X_banksy_lam{int(lam*10)}'
    sc.pp.neighbors(adata, use_rep=rep_key, key_added=f'nn_lam{int(lam*10)}')
    sc.tl.leiden(adata, neighbors_key=f'nn_lam{int(lam*10)}',
                 resolution=0.4, key_added=f'leiden_lam{int(lam*10)}')
    sq.pl.spatial_scatter(adata, color=f'leiden_lam{int(lam*10)}',
                           size=1.5, ax=ax, figsize=(4, 4))
    ax.set_title(f'λ = {lam}')
plt.suptitle('Effect of spatial context weight (λ) on tissue domains', y=1.02)
plt.tight_layout()
plt.show()

## 3. BANKSY: Full Implementation

The `banksy` package implements this with learned multi-scale neighborhood aggregation and additional features like NMF-based dimensionality reduction.

In [ ]:
try:
    from banksy.main import BanksyObject
    from banksy.banksy_utils.filter_utils import normalize_total, filter_hvg
    HAS_BANKSY = True
    print('BANKSY available')
except ImportError:
    HAS_BANKSY = False
    print('Install: pip install banksy-py')
    print('Using manual lambda sweep above as equivalent demonstration')

In [ ]:
if HAS_BANKSY:
    import scipy.sparse as sp
    
    # BANKSY needs raw (or normalized) expression and spatial coordinates
    coords = adata.obsm['spatial'].astype(float)
    expr = adata.X if not sp.issparse(adata.X) else adata.X.toarray()
    
    banksy_obj = BanksyObject(
        cell_by_gene=expr,
        cell_coordinates=coords,
    )
    
    # Compute the BANKSY matrix: augment expression with spatial neighbor averages
    # lambda_param: neighborhood contribution weight (0=expression only, 1=spatial only)
    # nbr_weight_decay: how neighbor weights fall off with distance
    banksy_obj.compute_banksyMatrix(
        lambda_param=0.2,
        nbr_weight_decay='scaled_gaussian',
        max_m=1,  # number of spatial smoothing scales
    )
    
    # PCA on BANKSY augmented matrix
    banksy_obj.run_pca(n_components=20, random_seed=42)
    
    # Leiden clustering on BANKSY embedding
    banksy_obj.run_leiden(
        resolution=0.5,
        nn_params={'n_neighbors': 15},
    )
    
    # Add to adata
    adata.obs['banksy_cluster'] = banksy_obj.adata.obs['banksy_labels'].values
    adata.obsm['X_banksy'] = banksy_obj.adata.obsm['banksy_pca']
    
    sq.pl.spatial_scatter(adata, color='banksy_cluster', size=1.5, figsize=(5, 5),
                           title='BANKSY spatial domains (λ=0.2)')
else:
    # Use the manual lambda=0.2 results from above
    adata.obs['banksy_cluster'] = adata.obs['leiden_lam2']
    sq.pl.spatial_scatter(adata, color='banksy_cluster', size=1.5, figsize=(5, 5),
                           title='Spatial domains (manual λ=0.2 augmentation)')

## 4. GraphST: Graph-based Spatial Transcriptomics

In [ ]:
try:
    from GraphST import GraphST
    HAS_GRAPHST = True
    print('GraphST available')
except ImportError:
    HAS_GRAPHST = False
    print('Install: pip install GraphST')

In [ ]:
if HAS_GRAPHST:
    # GraphST uses a graph attention autoencoder
    # platform: 'Visium' | 'Slide-seq' | 'MERFISH' | 'seqFISH'
    model = GraphST.GraphST(
        adata,
        platform='Visium',
        epochs=500,
        random_seed=42,
    )
    adata = model.train()  # adds adata.obsm['emb'] + adata.obs['mclust'] if R mclust available
    
    # Leiden on GraphST embedding
    sc.pp.neighbors(adata, use_rep='emb', key_added='graphst_nn')
    sc.tl.leiden(adata, neighbors_key='graphst_nn', resolution=0.4, key_added='leiden_graphst')
    
    sq.pl.spatial_scatter(adata, color='leiden_graphst', size=1.5, figsize=(5, 5),
                           title='GraphST spatial domains')
else:
    print('GraphST not installed. Comparing BANKSY (or λ=0.2) vs. expression-only:')
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    sq.pl.spatial_scatter(adata, color='leiden', size=1.5, ax=axes[0], figsize=(4, 4))
    axes[0].set_title('Expression-only Leiden')
    sq.pl.spatial_scatter(adata, color='banksy_cluster', size=1.5, ax=axes[1], figsize=(4, 4))
    axes[1].set_title('Spatially-aware clustering')
    plt.tight_layout()
    plt.show()

## 5. Evaluating Domain Quality

How do we know if spatially-aware clustering is actually better? Two approaches:
1. **Overlap with known anatomy** (when ground truth is available)
2. **Spatial coherence score** (are co-domain spots physically close?)

In [ ]:
# Spatial coherence: for each cluster, what fraction of spot pairs within the cluster
# are also spatial neighbors (within k=6 hops)?
from sklearn.metrics import adjusted_rand_score

# Compare clusters to known anatomical annotation (reference)
if 'cluster' in adata.obs:  # pre-annotated in the squidpy dataset
    ari_expr = adjusted_rand_score(adata.obs['cluster'], adata.obs['leiden'])
    ari_spatial = adjusted_rand_score(adata.obs['cluster'], adata.obs['banksy_cluster'])
    print(f'ARI vs. anatomy — expression-only Leiden: {ari_expr:.3f}')
    print(f'ARI vs. anatomy — spatially-aware:         {ari_spatial:.3f}')
    print('(Higher ARI = more consistent with known anatomical regions)')

In [ ]:
# Spatial fragmentation: count spatially isolated spots (not surrounded by same-cluster neighbors)
W = adata.obsp['spatial_connectivities'].toarray()

def spatial_fragmentation(labels, W):
    """Fraction of spots that are spatial outliers (none of their 6 neighbors share their label)"""
    isolated = 0
    for i, label in enumerate(labels):
        neighbor_labels = labels[W[i] > 0]
        if len(neighbor_labels) > 0 and not (neighbor_labels == label).any():
            isolated += 1
    return isolated / len(labels)

frag_expr = spatial_fragmentation(adata.obs['leiden'].values, W)
frag_spat = spatial_fragmentation(adata.obs['banksy_cluster'].values, W)
print(f'Fragmentation (lower = more spatially coherent):')
print(f'  Expression-only Leiden: {frag_expr:.3f}')
print(f'  Spatially-aware:        {frag_spat:.3f}')

---
## Summary

**The core insight:** Spatial domain detection = replace/augment PCA embedding with spatially-smoothed expression, then cluster as usual.

```python
# Manual BANKSY-style augmentation
W_norm = spatial_connectivity_matrix / row_sums
X_aug = (1 - lam) * X_pca + lam * (W_norm @ X_pca)  # lam = 0.2 typical
sc.pp.neighbors(adata, use_rep='X_aug')
sc.tl.leiden(adata)
```

| λ | Effect |
|---|--------|
| 0.0 | Pure expression-only (same as standard Leiden) |
| 0.2 | Gentle spatial smoothing; preserves fine expression differences |
| 0.5 | Equal weighting; good for layer detection |
| 0.8 | Strong spatial; useful for detecting broad tissue zones |

**Next:** SP03 — Sub-cellular resolution with seqFISH+ / Xenium.